In [18]:
import pandas as pd
import polars as pl
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report,f1_score
import xgboost as xgb
from sklearn.utils.class_weight import compute_sample_weight

In [2]:
df = pl.read_parquet("../data/03_processed/baseline_features_train.parquet").to_pandas()
df = df.dropna(subset='IncidentGrade')
label_map = {
    "FalsePositive" : 0,
    'BenignPositive' : 1,
    'TruePositive' : 2
}
df['target'] = df['IncidentGrade'].map(label_map)


X = df.drop(columns=['IncidentGrade','start_time','end_time','target','OrgId','IncidentId'])
y = df['target']

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)

print(f"Training set shape: {X_train.shape}")
print(f"Validation set shape: {X_test.shape}")
print(f"Target distribution:\n{y_train.value_counts(normalize=True) * 100}")

Training set shape: (358277, 31)
Validation set shape: (89570, 31)
Target distribution:
target
1    48.575823
0    30.119433
2    21.304745
Name: proportion, dtype: float64


In [3]:
rf_baseline = RandomForestClassifier(
    n_estimators=50,
    max_depth=10,
    random_state=42,
    n_jobs=-1
)
print("Training the Random Forest Model ...")
rf_baseline.fit(X_train,y_train)

print("Predicting on validation set ...")
y_pred = rf_baseline.predict(X_test)

print("\n--- Classification Report ---")
target_labels = ['False Positive (0)','Benign Positive (1)','True Positive (2)']
print(classification_report(y_test,y_pred,target_names=target_labels))

macro_f1 = f1_score(y_test, y_pred, average='macro')
print(f"\nCRITICAL BENCHMARK -> Baseline Macro-F1 Score: {macro_f1:.4f}")

Training the Random Forest Model ...
Predicting on validation set ...

--- Classification Report ---
                     precision    recall  f1-score   support

 False Positive (0)       0.81      0.60      0.69     26978
Benign Positive (1)       0.67      0.87      0.76     43509
  True Positive (2)       0.61      0.39      0.48     19083

           accuracy                           0.69     89570
          macro avg       0.70      0.62      0.64     89570
       weighted avg       0.70      0.69      0.68     89570


CRITICAL BENCHMARK -> Baseline Macro-F1 Score: 0.6430


## Baseline model on engineered features

In [4]:
df = pl.read_parquet("../data/03_processed/engineered_features_train.parquet").to_pandas()
df = df.dropna(subset='IncidentGrade')
df['target'] = df['IncidentGrade'].map(label_map)

X = df.drop(columns=['IncidentGrade','target','OrgId','IncidentId'])
y = df['target']

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)

print(f"Training set shape: {X_train.shape}")
print(f"Validation set shape: {X_test.shape}")
print(f"Target distribution:\n{y_train.value_counts(normalize=True) * 100}")

Training set shape: (358277, 37)
Validation set shape: (89570, 37)
Target distribution:
target
1    48.575823
0    30.119433
2    21.304745
Name: proportion, dtype: float64


In [5]:
rf_baseline = RandomForestClassifier(
    n_estimators = 50,
    max_depth=10,
    random_state=42,
    n_jobs = -1
)

print("Training the Random Forest Model ...")
rf_baseline.fit(X_train,y_train)

print("Predicting on validation set ...")
y_pred = rf_baseline.predict(X_test)

print("\n--- Classification Report ---")
print(classification_report(y_test,y_pred,target_names=target_labels))

macro_f1 = f1_score(y_test,y_pred,average='macro')
print(f"\nCRITICAL BENCHMARK -> Baseline Macro-F1 Score: {macro_f1:.4f}")

Training the Random Forest Model ...
Predicting on validation set ...

--- Classification Report ---
                     precision    recall  f1-score   support

 False Positive (0)       0.81      0.61      0.70     26978
Benign Positive (1)       0.67      0.87      0.76     43509
  True Positive (2)       0.62      0.40      0.49     19083

           accuracy                           0.69     89570
          macro avg       0.70      0.63      0.65     89570
       weighted avg       0.70      0.69      0.68     89570


CRITICAL BENCHMARK -> Baseline Macro-F1 Score: 0.6459


In [6]:
importance = pd.Series(rf_baseline.feature_importances_,index=X_train.columns)
print("Top 15 Most Important Features:")
print(importance.sort_values(ascending=False).head(15))

Top 15 Most Important Features:
evidence_per_second             0.089871
total_evidence_count            0.079487
unique_state_count              0.070856
unique_entitytype_count         0.067229
unique_countrycode_count        0.065187
unique_city_count               0.063227
is_multinational                0.061812
unique_sha256_count             0.038290
unique_filename_count           0.035781
unique_url_count                0.034859
ips_per_device                  0.032254
unique_accountobjectid_count    0.029955
unique_accountupn_count         0.029347
unique_applicationname_count    0.027688
unique_applicationid_count      0.027437
dtype: float64


## Version two of improved baseline model

In [7]:
df = pl.read_parquet('../data/03_processed/version_two_engineered_features_train.parquet').to_pandas()
df = df.dropna(subset=['IncidentGrade'])

df['target'] = df['IncidentGrade'].map(label_map)

X = df.drop(columns=['IncidentId','OrgId','IncidentGrade','target'])
y = df['target']

X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=42,stratify=y)

print(f"Shape of training data : {X_train.shape}")
print(f"Shape of test data : {X_test.shape}")
print(f"Target distribution in training data : {y_train.value_counts(normalize=True) * 100}")

Shape of training data : (358277, 55)
Shape of test data : (89570, 55)
Target distribution in training data : target
1    48.575823
0    30.119433
2    21.304745
Name: proportion, dtype: float64


In [8]:
rf_baseline = RandomForestClassifier(
    n_estimators=50,
    max_depth=10,
    random_state=42,
    n_jobs= -1 
)

print("Training the Random Forest Classifier ...")
rf_baseline.fit(X_train,y_train)

print("Predicting on Validation set ...")
y_pred = rf_baseline.predict(X_test)

print("\n --- Classification Report ---")
print(classification_report(y_test,y_pred,target_names=target_labels))

macro_f1 = f1_score(y_test,y_pred,average='macro')
print(f"\nCRITICAL BENCHMARK -> Baseline Macro-F1 Score: {macro_f1:.4f}")

Training the Random Forest Classifier ...
Predicting on Validation set ...

 --- Classification Report ---
                     precision    recall  f1-score   support

 False Positive (0)       0.82      0.60      0.69     26978
Benign Positive (1)       0.67      0.88      0.76     43509
  True Positive (2)       0.61      0.40      0.48     19083

           accuracy                           0.69     89570
          macro avg       0.70      0.63      0.64     89570
       weighted avg       0.70      0.69      0.68     89570


CRITICAL BENCHMARK -> Baseline Macro-F1 Score: 0.6447


In [9]:
importance = pd.Series(rf_baseline.feature_importances_,index = X_train.columns)
print("Top 15 Most Important Features:")
print(importance.sort_values(ascending=False).head(15))

Top 15 Most Important Features:
total_evidence_count          0.084056
evidence_per_second           0.078589
is_multinational              0.072003
unique_entitytype_count       0.064801
unique_state_count            0.062036
unique_countrycode_count      0.058722
unique_city_count             0.051875
unique_accountname_count      0.039121
devices_per_account           0.034301
cat_Execution                 0.029651
unique_url_count              0.029252
ips_per_device                0.028770
unique_filename_count         0.027619
unique_applicationid_count    0.027370
incident_duration_seconds     0.024246
dtype: float64


In [19]:
print("Computing class weight for each class ...")
sample_weights = compute_sample_weight(class_weight='balanced' , y = y_train)

xgb_model = xgb.XGBClassifier(
    n_estimators  = 200,
    max_depth = 6,
    learning_rate = 0.1,
    objective = 'multi:softmax',
    num_class = 3,
    random_state = 42,
    n_jobs = -1
)


print('Training XGboost classiffier ...')
xgb_model.fit(X_train,y_train, sample_weight=sample_weights)

print('Predicting on validation set ...')
y_pred = xgb_model.predict(X_test)

print('\n --- Classification Report ---')
print(classification_report(y_test,y_pred,target_names=target_labels))

macro_f1 = f1_score(y_test,y_pred,average = 'macro')
print(f"\nCRITICAL BENCHMARK -> Baseline Macro-F1 Score: {macro_f1:.4f}")

importance = pd.Series(xgb_model.feature_importances_,index = X_train.columns)
print("Top 15 Most Important Features:")
print(importance.sort_values(ascending=False).head(15))

Computing class weight for each class ...
Training XGboost classiffier ...
Predicting on validation set ...

 --- Classification Report ---
                     precision    recall  f1-score   support

 False Positive (0)       0.74      0.68      0.71     26978
Benign Positive (1)       0.76      0.65      0.70     43509
  True Positive (2)       0.47      0.68      0.56     19083

           accuracy                           0.67     89570
          macro avg       0.66      0.67      0.66     89570
       weighted avg       0.69      0.67      0.67     89570


CRITICAL BENCHMARK -> Baseline Macro-F1 Score: 0.6564
Top 15 Most Important Features:
unique_countrycode_count        0.180754
unique_state_count              0.110351
cat_Execution                   0.072935
cat_Exfiltration                0.064844
cat_SuspiciousActivity          0.063744
unique_entitytype_count         0.051230
unique_applicationid_count      0.042063
evidence_per_second             0.033860
unique_detector

In [23]:
df = pl.read_parquet("../data/03_processed/version_two_engineered_features_train.parquet").to_pandas()
df['text_AlertTitle'].head()

0                 2 3 278
1                  3659 2
2    1 12238 8823 6125 18
3                       0
4                     289
Name: text_AlertTitle, dtype: str